# 04 · LocalGLMnet: an interpretable network
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/actuarialsociety/deep-actuarial-science/blob/main/notebooks/04_localglmnet.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/actuarialsociety/deep-actuarial-science/blob/main/notebooks/04_localglmnet.ipynb) [![Open In SageMaker Studio](https://img.shields.io/badge/Open%20in-SageMaker%20Studio-FF9900)](https://console.aws.amazon.com/sagemaker/home#/studio-landing)

$\log\lambda(x)=\beta_0+\sum_j\beta_j(x)\,x_j$: the network outputs one coefficient per covariate, so it reads like a GLM.

_Runs anywhere: **Colab** opens on a GPU runtime (T4) and the first cell stops on a CPU runtime; on **Kaggle** turn on *GPU T4* and *Internet* in the notebook settings; in **SageMaker Studio** (the badge opens the console) or **Azure ML**, pick a GPU instance, clone the repo (Git → Clone) or upload this notebook, and run all: the first cell installs what it needs. Any other Jupyter works the same way._

In [ ]:
# Setup: runs anywhere Jupyter runs. Opened on its own (Colab, Kaggle, SageMaker, Azure ML,
# any hosted Jupyter), this cell clones and installs the repo; in a clone or the container it
# uses that checkout.
import os, sys, importlib.util
# Colab must run on a GPU runtime: stop here otherwise (set DAS_ALLOW_CPU=1 to override).
if "google.colab" in sys.modules and os.environ.get("DAS_ALLOW_CPU") != "1":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "This notebook needs a GPU runtime. In Colab: Runtime > Change runtime type > "
            "T4 GPU > Save, then Runtime > Run all. (To run on CPU anyway: "
            "import os; os.environ['DAS_ALLOW_CPU'] = '1')"
        )
    print("GPU:", torch.cuda.get_device_name(0))

REPO_DIR = next((d for d in (os.getcwd(), os.path.dirname(os.getcwd()))
                 if os.path.isdir(os.path.join(d, "src", "deep_actuarial_science"))), None)
if REPO_DIR is None:
    if sys.version_info < (3, 10):
        raise RuntimeError("Needs Python 3.10+: choose a newer kernel or image")
    # a writable, persistent home: Colab, Kaggle, SageMaker notebook instances, else ~
    base = next(d for d in ("/content", "/kaggle/working", os.path.expanduser("~/SageMaker"),
                            os.path.expanduser("~")) if os.path.isdir(d) and os.access(d, os.W_OK))
    REPO_DIR = os.path.join(base, "deep-actuarial-science")
    if not os.path.exists(REPO_DIR):  # Kaggle: turn on Internet in the notebook settings
        !git clone -q https://github.com/actuarialsociety/deep-actuarial-science {REPO_DIR}
    if importlib.util.find_spec("deep_actuarial_science") is None:
        !pip install -q -e {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, "src")
os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

from deep_actuarial_science.notebook import run, load, show, leaderboard, parameters, mlflow_ui, device_summary
device_summary()
FAST = os.environ.get("DAS_ENV") == "smoke"   # CI runs every notebook in smoke mode


In [ ]:
run("frequency", models=["localglmnet"])
leaderboard("frequency", ["localglmnet"])

In [ ]:
load("localglmnet.freq_reconciliation")

## Attention weights and variable selection
`RandN` is pure noise: its attention weights define the band (shaded) inside which a covariate is not significant.

In [ ]:
show("frequency", "localglmnet", ["attention", "importance"], width=820)

In [ ]:
m = load("localglmnet.freq_model"); test = load("freq_test")
att = m.attentions(test); band = 2.576 * att["RandN"].std()
(att.abs().mean() / band).sort_values().rename("mean |beta| / noise band").to_frame()

## Exercise
Drop the covariates that sit inside the noise band (edit `frequency.models.localglmnet.features`) and re-run. Does the test deviance change?

## Explore the runs in MLflow
Parameters, metrics, figures and the benchmark table of every run above. MLflow runs on the notebook's machine: in Colab it opens below and in a new tab through Colab's proxy; in SageMaker and Azure ML through Jupyter's proxy; on Kaggle, which exposes no ports, query the runs with `mlflow.search_runs` instead.

In [ ]:
mlflow_ui()